# Qt–Ft Agglomeration Simulation — Run Notebook

Coarse-grained ReaDDy2 Brownian-dynamics simulation of Qt encapsulin and ferritin (Ft) agglomeration.

**This notebook only runs simulations.** All parameters live in the single **Configuration** cell; the single **Run** cell then executes any combination of:

- `RUN_MODE = "single"` (one trajectory) or `"ensemble"` (multi-replica)
- `ENABLE_DEAGG = False` (plain agglomeration) or `True` (agglomeration ↔ deagglomeration cycling)

Plotting and analysis live in the separate `Plot_Ensemble_Results_*.ipynb` notebooks.

## 1. Imports and Setup

In [1]:
import os
import sys
import readdy

# qtft package. Plotting lives in the Plot_Ensemble_Results_* notebooks, so it is
# deliberately not imported here; `analysis` is only used for the optional summary/XYZ export.
import qtft as sim
import qtft.analysis as analysis
from qtft import EnsembleSimulation

print(f"ReaDDy: {readdy.__version__}")
print(f"Python: {sys.version}")

ReaDDy: 2.0.13-5
Python: 3.10.19 | packaged by conda-forge | (main, Oct 22 2025, 22:29:10) [GCC 14.3.0]


## 2. Module Reload Helper

In [2]:
import importlib
import qtft, qtft.config, qtft.system, qtft.engine, qtft.analysis, qtft.ensemble

# Reload submodules in dependency order (restart the kernel for deep changes).
for _m in (qtft.config, qtft.system, qtft.engine, qtft.analysis, qtft.ensemble, qtft):
    importlib.reload(_m)

sim = qtft
analysis = qtft.analysis
from qtft import EnsembleSimulation
print("Modules reloaded")

Modules reloaded


## 3. Configuration — all parameters

In [3]:
# ============================================================
# SIMULATION CONFIGURATION  —  all parameters live here
# ============================================================
# Edit parameters, run this cell, then run the "Run" cell below.

# ----- What to run -----
RUN_MODE     = "ensemble"   # "single" (one trajectory) or "ensemble" (multi-replica)
ENABLE_DEAGG = False      # False = plain agglomeration; True = agglomeration <-> deagglomeration cycling

# ----- Output locations -----
SINGLE_RUN_ROOT = "Simulation_Files_Single_Runs"  # parent folder for single runs
ENSEMBLE_ROOT   = "Simulation_Files_Ensembles"    # parent folder for ensembles

# ----- Output toggles -----
SAVE_CONFIG = True    # write <name>_config.json next to the trajectory (single runs; ensembles always save it)
EXPORT_XYZ  = True   # also export an OVITO-friendly .xyz after the run

# ----- Ensemble settings (used when RUN_MODE == "ensemble") -----
N_REPLICAS          = 10
PARALLEL            = True
N_WORKERS           = 10
EQUILIBRATION_STEPS = 10000   # WCA, reaction-free relaxation before production (all run types)

# ----- Deagglomeration / cycling settings (used when ENABLE_DEAGG) -----
KOFF            = 1e-5        # per-edge bond-breaking rate (1/ns); must be > 0 to dissolve clusters
AGG_STEPS       = 1_000_000   # steps of each agglomeration phase
DEAGG_STEPS     = 1_000_000   # steps of each deagglomeration phase
N_CYCLES        = 1           # number of agg->deagg cycles (total phases = 2 * N_CYCLES)
AGG_POTENTIAL   = "WCA"        # potential during agglomeration ("LJ" attractive / "WCA" repulsive)
DEAGG_POTENTIAL = "WCA"       # potential during deagglomeration

# ============================================================
# Build the configuration
# ============================================================
config = sim.SimulationConfig(
    # ----- Particle properties -----
    qt=sim.ParticleConfig(
        name="Qt",
        radius=21.0,             # nm (encapsulin)
        diffusion=0.5,           # nm^2/ns
        cluster_diffusion=0.3,   # nm^2/ns (when bound in a cluster)
    ),
    ft=sim.ParticleConfig(
        name="Ft",
        radius=6.0,              # nm (ferritin)
        diffusion=1.0,           # nm^2/ns
        cluster_diffusion=0.7,   # nm^2/ns (when bound in a cluster)
    ),

    # ----- Topology / binding -----
    topology=sim.TopologyConfig(
        name="QtFt_Cluster",
        binding_radius=27.25,    # nm (~ r_Qt + r_Ft + buffer)
        kon=0.001,                 # microscopic binding rate (1/ns)
        k_bond=10,               # kJ/(mol*nm^2) bond stiffness
        ft_monovalent=False,     # True -> Ft caps at 1 bond (single-Qt-star clusters); adds _FtMono tag
    ),

    # ----- Lennard-Jones (per-pair well depths, kJ/mol) -----
    # Cluster/mixed-state values cascade from the free-free values unless set.
    # Set any epsilon to 0 to disable that interaction entirely.
    lj=sim.LennardJonesConfig(
        epsilon_QtQt=1.5,        # Qt-Qt
        epsilon_FtFt=1.5,        # Ft-Ft
        epsilon_QtFt=3.0,        # Qt-Ft
        epsilon_QtCQtC=None,     # defaults to epsilon_QtQt
        epsilon_FtCFtC=None,     # defaults to epsilon_FtFt
        epsilon_QtCFtC=None,     # defaults to epsilon_QtFt
        potential_type="LJ",     # "LJ" (attractive, production) or "WCA" (repulsive)
    ),

    # ----- Simulation box -----
    box_size=(500, 500, 500),    # nm
    periodic_boundary=True,
    temperature=300.0,           # K

    # ----- Integration -----
    timestep=5e-2,               # ns (0.05 ns = 50 ps)
    n_steps=2_000_000,           # total steps for a plain run (ignored when ENABLE_DEAGG)

    # ----- Recording -----
    record_stride=100,                  # save trajectory every N steps
    observable_stride=100,              # record observables every N steps
    particles_observable_stride=1000,   # per-particle positions cadence (None to disable, saves disk)

    # ----- Particle counts -----
    n_qt=200,
    n_ft=400,

    # ----- Execution -----
    kernel="CPU",
    n_threads=4,
    rng_seed=22,

    # output_file is auto-generated from the parameters (see qtft.format_param_string).
)

# ----- Apply deagglomeration cycling (centralizes the phase knobs set above) -----
if ENABLE_DEAGG:
    config.topology.koff = KOFF
    config.phases = sim.make_agg_deagg_phases(
        agg_steps=AGG_STEPS,
        deagg_steps=DEAGG_STEPS,
        n_cycles=N_CYCLES,
        agg_potential=AGG_POTENTIAL,
        deagg_potential=DEAGG_POTENTIAL,
    )

config.print_summary()

SIMULATION CONFIGURATION

Particles:
  Qt: r=21.0 nm, D=0.5 nm²/ns (cluster: D=0.3)
  Ft: r=6.0 nm, D=1.0 nm²/ns (cluster: D=0.7)
  Counts: 200 Qt + 400 Ft = 600 total

Topology:
  Binding radius: 27.25 nm
  Binding rate (kon): 0.001 nm³/(ns·part)
  Bond-breaking rate (koff): 0.0 /(edge·ns)
  Bond stiffness: 10 kJ/(mol·nm²)
  Equilibrium bond length: 27.0 nm

Lennard-Jones:
  Potential type: LJ
  Cutoff factor: 2.500
  ε Qt-Qt: 1.5 kJ/mol
  ε Ft-Ft: 1.5 kJ/mol
  ε Qt-Ft: 3.0 kJ/mol
  Cluster/mixed ε: same as free (default)

Simulation:
  Box: 500 × 500 × 500 nm
  Temperature: 300.0 K
  Equilibration potential: WCA
  Timestep: 0.05 ns (50.00 ps)
  Steps: 2,000,000 (100.0 µs total)
  Output: 200Qt_400Ft_LJ_eQQ1.5_eFF1.5_eQF3_kon0.001_dt50ps_100us.h5


## 4. Run

In [6]:
# ============================================================
# RUN  —  dispatches on RUN_MODE and ENABLE_DEAGG (set in the config cell)
# ============================================================

# Auto-named basename from the parameters (idempotent: safe to re-run this cell).
base_name = sim.format_param_string(config) + ".h5"

if RUN_MODE == "single":
    # Collect all output for this run in its own subfolder under SINGLE_RUN_ROOT.
    RUN_DIR = os.path.join(SINGLE_RUN_ROOT, base_name[:-3])
    config.output_file = os.path.join(RUN_DIR, base_name)

    result = sim.run_one(config, equilibration_steps=EQUILIBRATION_STEPS)

    if config.phases:
        # Phased run: result is a list of per-phase dicts; the stitched whole-cycle
        # trajectory path is under result[0]["combined"].
        combined = result[0].get("combined")
        export_src = combined
        print(f"Phased run complete ({len(result)} phases). Combined trajectory: {combined}")
    else:
        # Plain run: result is a readdy.Trajectory.
        export_src = config.output_file
        analysis.print_analysis_summary(config.output_file, config)

    if SAVE_CONFIG:
        cfg_path = config.output_file[:-3] + "_config.json"
        config.save_json(cfg_path)
        print(f"Saved config: {cfg_path}")

    if EXPORT_XYZ and export_src and os.path.exists(export_src):
        xyz_path = export_src.replace(".h5", ".xyz")
        analysis.convert_h5_to_xyz(export_src, xyz_path, config, overwrite=True)
        print(f"Exported XYZ: {xyz_path}")

elif RUN_MODE == "ensemble":
    ensemble = EnsembleSimulation(
        base_config=config,
        n_replicas=N_REPLICAS,
        base_dir=ENSEMBLE_ROOT,
    )
    print(f"Seeds: {ensemble.seeds}")
    ensemble.run_local(
        parallel=PARALLEL,
        n_workers=N_WORKERS,
        overwrite=True,
        equilibration_steps=EQUILIBRATION_STEPS,
    )
    ensemble.print_summary()

    if EXPORT_XYZ:
        rep_cfg = ensemble.replica_configs[0]
        if config.phases:
            src = os.path.join(rep_cfg.phase_base_dir, "trajectory_combined.h5")
        else:
            src = rep_cfg.output_file
        if os.path.exists(src):
            xyz_path = src.replace(".h5", ".xyz")
            analysis.convert_h5_to_xyz(src, xyz_path, rep_cfg, overwrite=True)
            print(f"Exported XYZ (replica_000): {xyz_path}")

else:
    raise ValueError(f"RUN_MODE must be 'single' or 'ensemble', got {RUN_MODE!r}")

✓ Ensemble created: 200Qt_400Ft_LJ_eQQ1.5_eFF1.5_eQF3_kon0.001_dt50ps_0.10us
  Output directory: Simulation_Files_Ensembles/200Qt_400Ft_LJ_eQQ1.5_eFF1.5_eQF3_kon0.001_dt50ps_0.10us/
  Replicas: 10
Seeds: [1950464737, 2539317803, 3333880692, 2150020773, 2633213151, 1529125172, 4010032112, 3532649711, 2858885004, 2799241814]
✓ Configuration saved to Simulation_Files_Ensembles/200Qt_400Ft_LJ_eQQ1.5_eFF1.5_eQF3_kon0.001_dt50ps_0.10us/configs/config_000.json
✓ Configuration saved to Simulation_Files_Ensembles/200Qt_400Ft_LJ_eQQ1.5_eFF1.5_eQF3_kon0.001_dt50ps_0.10us/configs/config_001.json
✓ Configuration saved to Simulation_Files_Ensembles/200Qt_400Ft_LJ_eQQ1.5_eFF1.5_eQF3_kon0.001_dt50ps_0.10us/configs/config_002.json
✓ Configuration saved to Simulation_Files_Ensembles/200Qt_400Ft_LJ_eQQ1.5_eFF1.5_eQF3_kon0.001_dt50ps_0.10us/configs/config_003.json
✓ Configuration saved to Simulation_Files_Ensembles/200Qt_400Ft_LJ_eQQ1.5_eFF1.5_eQF3_kon0.001_dt50ps_0.10us/configs/config_004.json
✓ Configur

 96%|█████████▌| 962/1000 [00:10<00:00, 87.52it/s]


EQUILIBRATION COMPLETE
  Retrieved 200 Qt + 400 Ft positions

✓ Species: Qt, Ft, QtC, FtC
✓ LJ potentials (10 registered): ε_QQ=1.5, ε_FF=1.5, ε_QF=3.0
✓ Topology 'QtFt_Cluster': k_bond=10; 4 binding spatial reactions (kon=0.001, binding_radius=27.25 nm, ft_monovalent=False)
✓ System created: 500×500×500 nm box


 96%|█████████▌| 961/1000 [00:10<00:00, 83.77it/s]

✓ Observables registered (stride=100, forces/virial stride=10000, particles observable stride=1000)
✓ Simulation created: CPU kernel, 4 threads
✓ Placed 200 Qt (provided) + 400 Ft (provided) particles



...


  Composition: 100%|██████████| 1/1 [00:00<00:00, 3625.15frame/s]

    ✓ morphology / spatial / contacts / composition

  Replica 9 (10/10):



  Composition: 100%|██████████| 1/1 [00:00<00:00, 3876.44frame/s]

    ✓ morphology / spatial / contacts / composition

Processing structural data...
  Computing size fractions...


    ✓ Size fractions (5 categories)
✓ Structural statistics computed
✓ Saved statistics to Simulation_Files_Ensembles/200Qt_400Ft_LJ_eQQ1.5_eFF1.5_eQF3_kon0.001_dt50ps_0.10us/ensemble_statistics.json
✓ Saved configuration to Simulation_Files_Ensembles/200Qt_400Ft_LJ_eQQ1.5_eFF1.5_eQF3_kon0.001_dt50ps_0.10us/ensemble_config.json
✓ Saved structural data to Simulation_Files_Ensembles/200Qt_400Ft_LJ_eQQ1.5_eFF1.5_eQF3_kon0.001_dt50ps_0.10us/ensemble_structural.npz
✓ Saved ensemble state to Simulation_Files_Ensembles/200Qt_400Ft_LJ_eQQ1.5_eFF1.5_eQF3_kon0.001_dt50ps_0.10us/ensemble_state.json
✓ Saved ensemble state to Simulation_Files_Ensembles/200Qt_400Ft_LJ_eQQ1.5_eFF1.5_eQF3_kon0.001_dt50ps_0.10us/ensemble_state.json

ENSEMBLE COMPLETE
Output directory: Simulation_Files_Ensembles/200Qt_400Ft_LJ_eQQ1.5_eFF1.5_eQF3_kon0.001_dt50ps_0.10us/
Analysis files saved for plotting


ENSEMBLE SUMMARY (N=10 replicas)

Time Series Metrics (final values):
  Bonds:              2.5 ± 1.4
  Clusters:    

## 5. Cluster (SLURM) execution — optional

For HPC runs, generate SLURM job-array scripts instead of running locally. This builds an `EnsembleSimulation` from the **Configuration** cell above (set the params there first), then writes `submit_ensemble.slurm` and `submit_analysis.slurm` into the ensemble directory. Nothing runs locally — submit them with `sbatch`.

In [ ]:
# ============================================================
# OPTIONAL: generate SLURM scripts for cluster execution
# ============================================================
# Builds the ensemble from the config above; does not run anything locally.
ensemble = EnsembleSimulation(
    base_config=config,
    n_replicas=N_REPLICAS,
    base_dir=ENSEMBLE_ROOT,
)

# ----- SLURM script for running replica simulations -----
ensemble.generate_slurm_scripts(
    # --- SLURM job settings ---
    partition="cm4_tiny",        # (required, str) SLURM partition name
    cluster="cm4",               # (optional, str) SLURM cluster name, None to omit
    qos="cm4_tiny",              # (optional, str) Quality of service, None to omit
    time="08:00:00",             # (optional, str) Wall time limit per replica (HH:MM:SS)
    cpus_per_task=12,            # (optional, int) CPUs per replica
    memory="32G",                # (optional, str) Memory per replica

    # --- Conda environment ---
    conda_base="<YOUR_CONDA_PATH>",  # (required, str) Full path to conda installation, e.g. "/home/user/miniconda3"
    conda_env="readdy",              # (optional, str) Name of conda environment with ReaDDy

    # --- Paths ---
    scripts_dir="~/Readdy_Simulations",  # (optional, str) Directory where Python scripts are located

    # --- Email notifications ---
    mail_user=None,              # (optional, str) Email for notifications, e.g. "user@example.com"
    mail_type="ALL",             # (optional, str) When to send emails: NONE, BEGIN, END, FAIL, ALL
)

# ----- SLURM script for post-simulation analysis -----
ensemble.generate_analysis_slurm_script(
    # --- SLURM job settings ---
    partition="cm4_tiny",        # (required, str) SLURM partition name
    cluster="cm4",               # (optional, str) SLURM cluster name, None to omit
    qos="cm4_tiny",              # (optional, str) Quality of service, None to omit
    time="04:00:00",             # (optional, str) Wall time limit (HH:MM:SS)
    cpus_per_task=4,             # (optional, int) CPUs for parallel analysis
    memory="32G",                # (optional, str) Memory allocation

    # --- Conda environment ---
    conda_base="<YOUR_CONDA_PATH>",  # (required, str) Full path to conda installation, e.g. "/home/user/miniconda3"
    conda_env="readdy",              # (optional, str) Name of conda environment with ReaDDy

    # --- Paths and analysis settings ---
    scripts_dir="~/Readdy_Simulations",  # (optional, str) Directory where Python scripts are located
    stride=10,                           # (optional, int) Analyze every Nth frame for structural analysis

    # --- Email notifications ---
    mail_user=None,              # (optional, str) Email for notifications, e.g. "user@example.com"
    mail_type="ALL",             # (optional, str) When to send emails: NONE, BEGIN, END, FAIL, ALL
)